# Benchmark — Feature Selection & Model Comparison

**Classification Transcriptomic with XAI**

This notebook runs the benchmark:
1. Load a gene expression dataset (CuMiDa format: `samples, type, probe_1, …`)
2. Apply each enabled feature selection method independently (Raw / mRMR / Boruta)
3. Benchmark all enabled models with Stratified K-Fold CV on each reduced dataset
4. **Run SHAP** on each trained model → plots + gene-level importance (probe→gene
   names from the dataset's local `annotation_file`)
5. Save per-fold metrics, summaries, trained models, and SHAP outputs to `outputs/`

Configure which datasets / methods / models run by editing the YAML files in `configs/`.

> Interpretation (association/tree rules) consumes the SHAP top genes produced here.

In [2]:
import sys
import warnings
from pathlib import Path

# Make src.* importable from the notebooks/ directory
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", message=".*does not have many workers.*")

import numpy as np
import pandas as pd

from src.pipeline import GeneExpressionPipeline

## Configuration

Set **`RUN_MODE`** to pick how the benchmark runs:

| Mode | Datasets benchmarked |
|------|----------------------|
| `"single"`   | one dataset (`DATASET`) |
| `"multiple"` | many datasets (`DATASETS`, or `None` = all enabled in `datasets.yaml`) |

Which FS methods / models / SHAP top-k run is controlled by the YAML files in `configs/`.

In [3]:
# ── Run mode ──────────────────────────────────────────────────────────────
# "single"   → benchmark ONE dataset (DATASET).
# "multiple" → benchmark MANY datasets (DATASETS; None = all enabled).
RUN_MODE = "multiple"            # "single" | "multiple"

# ── Datasets ────────────────────────────────────────────────────────────────
DATASET  = "GEO-Breast-20711"
DATASETS = []   # e.g. ["GEO-Breast-20711", "GEO-Breast-20711-gene"] or None

# ── W&B (optional) ────────────────────────────────────────────────────────────
USE_WANDB = True

## 1. Initialise Pipeline

In [4]:
pipe = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs',
    use_wandb=USE_WANDB,
)

# Show enabled datasets / FS methods / models
pipe.print_summary()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Bao Nguyen\_netrc.
wandb: Currently logged in as: myb2207544 (myb2207544-can-tho-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Configuration Summary                                                                                           │
│ Classification Transcriptomic with XAI                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

    Project root : C:\Users\Bao Nguyen\LearningWithClaude\LV_code

    Config root  : C:\Users\Bao Nguyen\LearningWithClaude\LV_code\configs

                               Datasets (12 enabled)                                
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ status ┃                 dataset ┃                                   description ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      ✓ │        GEO-Breast-20711 │       Breast cancer – IHC subtypes (GSE20711) │
│      ✓ │  GEO-Mesothelioma-29354 │      Pleural mesothelioma subtypes (GSE29354) │
│      ✓ │        GEO-Kidney-36895 │    Clear-cell renal cell carcinoma (GSE36895) │
│      ✓ │         GEO-Colon-68468 │  Colon cancer molecular dissection (GSE68468) │
│      ✓ │   GEO-MultiCancer-68606 │ Multi-cancer interlaboratory study (GSE68606) │
│      ✓ │       CuMiDa-Ovary-6008 │                      Ovarian cancer (GSE6008) │
│      ✓ │      CuMiDa-Brain-15824 │               Brain tumor subtypes (GSE15824) │
│      ✓ │     CuMiDa-Breast-26304 │                      Breast cancer (GSE26304) │
│      ✓ │   CuMiDa-Leukemia-28497 │                  Leukemia subtypes (GSE28497) │
│      ✓ │ CuMiDa-Colorectal-41657 │                  Colorectal cancer (GSE41657) │
│      ✓ │     CuMiDa-Breast-45827 │             Breast cancer subtypes (GSE45827) │
│      ✓ │      CuMiDa-Brain-50161 │               Brain tumor subtypes (GSE50161) │
└────────┴─────────────────────────┴───────────────────────────────────────────────┘

    Models (7): nb, knn, svm, rf, dt, xgboost, ann

    Feature selection (4): raw, mrmr_k50, mrmr_k75, boruta

    Cross-validation : max 5-fold stratified

    W&B project      : transcriptomic_xai

## 2. Rare-class k sweep — full baseline + train/test-split baseline

Sweeps `feature_selection.yaml -> rare_class_k_values` for **both** baseline flows —

- **CV/full-data** (`pipe_full.run_benchmark(..., k_values=...)`) — writes to `outputs_baseline_full/k{k}/{dataset}/...`,
  raw + boruta + mrmr_k50/mrmr_k75, all enabled models.
- **Train/test-split** (`pipe.run_baseline_split(..., k_values=...)`) — writes to `outputs_baseline_split/k{k}/{dataset}/...`,
  boruta + mrmr_k50/mrmr_k75 only (no raw), all enabled models. This is also the **source of truth**
  `pipe.run_rule_extraction_holdout()` (notebook 04) reads its split + FS artifacts from for the same k.


In [ ]:
benchmark_datasets = [DATASET] if RUN_MODE == "single" else DATASETS
K_VALUES = pipe.config_loader.load_feature_selection_config().get("rare_class_k_values", [2,3, 4, 5, 6])

In [ ]:
sweep_datasets = benchmark_datasets or pipe.config_loader.get_enabled_datasets()
print(f"Sweeping k={K_VALUES} for {len(sweep_datasets)} dataset(s) -> outputs_baseline_full/")

pipe_full = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs_baseline_full',
    use_wandb=USE_WANDB,
)

full_sweep_results = pipe_full.run_benchmark(sweep_datasets, k_values=K_VALUES)

In [ ]:
# Train/test-split baseline sweep -- also the split+FS source for notebook 04's rule extraction.
split_sweep_results = pipe.run_baseline_split(sweep_datasets, k_values=K_VALUES)


In [ ]:
# Verify every (dataset, k, fs_method, model) combo from the split-baseline
# sweep above actually has its artifacts on disk, and rerun only what's
# missing. run_baseline_split() has no built-in resume, so a run
# interrupted partway (power loss, KeyboardInterrupt, ...) would otherwise
# leave outputs_baseline_split/ silently incomplete -- notebook 04 reads it
# as its source of truth for rule extraction, so this must be verified
# (not just assumed complete) before moving on.
holdout_cfg = pipe.config_loader.load_yaml("holdout.yaml")["holdout"]
split_root = Path(holdout_cfg["split_baseline"].get("baseline_split_root", "outputs_baseline_split"))
if not split_root.is_absolute():
    split_root = pipe.config_loader.project_root / split_root

fs_methods = holdout_cfg["batch"].get("fs_methods", ["boruta", "mrmr_k50", "mrmr_k75"])
enabled_models = pipe.config_loader.get_enabled_models()

missing = {}  # (dataset, k) -> list of missing "fs_method/model" combos
for k in K_VALUES:
    for ds in sweep_datasets:
        gaps = [
            f"{fs}/{model}"
            for fs in fs_methods
            for model in enabled_models
            if not (split_root / f"k{k}" / ds / fs / model / "models" / f"{model}.joblib").exists()
        ]
        if gaps:
            missing[(ds, k)] = gaps

if missing:
    print(f"Found {len(missing)} incomplete (dataset, k) combo(s) under {split_root}:")
    for (ds, k), gaps in missing.items():
        print(f"  k={k} {ds}: missing {gaps}")
    for ds, k in missing:
        print(f"Rerunning k={k} {ds}...")
        pipe.run_baseline_split([ds], k_values=[k])
else:
    print("All (dataset, k, fs_method, model) combos from the split-baseline sweep are present.")


Interrupt due to electrictory --> rerun k=2 for baseline_full:  CuMiDa-Colorectal-41657, CuMiDa-Breast-45827,  CuMiDa-Brain-50161

In [ ]:
sweep_datasets = [DATASET] if RUN_MODE == "single" else (DATASETS or None)

pipe_full = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs_baseline_full',
    use_wandb=True,
)
sweep_datasets = sweep_datasets or pipe_full.config_loader.get_enabled_datasets()
K_VALUES = pipe_full.config_loader.load_feature_selection_config().get("rare_class_k_values", [2, 3, 4, 5, 6])

# 1) Rerun k=2
k2_pending = ["CuMiDa-Colorectal-41657", "CuMiDa-Brain-50161", "CuMiDa-Breast-45827"]
pipe_full.run_benchmark(k2_pending, k_values=[2])

# 2) Continue with >=k3
remaining_k_values = [k for k in K_VALUES if k >= 3]
pipe_full.run_benchmark(sweep_datasets, k_values=remaining_k_values)